In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.visualisation.spatial_visualisation import SpatialVisualisation
from quinex_application.postprocessing.data_processing import apply_iqr_filter

In [ ]:
visualisation = SpatialVisualisation()

### **Import Results**

#### **LCOE**

In [ ]:
cost_lcoe = pd.read_excel("../data/visualisation_files/unit_converted/df_technologies_lcoe2024.xlsx")
cost_lcoe = cost_lcoe[cost_lcoe["entity_base_mapping"].isin(['PV'])]

In [ ]:
cost_lcoe = visualisation.convert_spatial(cost_lcoe, spatial_col="spatial_scope")
print(cost_lcoe["entity_base_mapping"].value_counts())

In [ ]:
cost_lcoe = cost_lcoe[cost_lcoe["value_converted"] < 510]
len(cost_lcoe)

#### **Solar Radiation**

In [ ]:
# import rasterio

# # GeoTIFF öffnen
# src = rasterio.open("../data/visualisation_files/ghi/GHI.tif")
# coords = zip(
#     cost_lcoe["longitude"],
#     cost_lcoe["latitude"]
# )

# cost_lcoe["solar_radiation"] = [
#     val[0] for val in src.sample(coords)
# ]

# def is_country_only(row):
#     text = row["spatial_scope"]["text"].strip().lower()
#     country = row["spatial_scope"]["country"].strip().lower()

#     return (
#         text == country
#         or text == f"in {country}"
#     )

# mask = cost_lcoe.apply(is_country_only, axis=1)

# cost_lcoe_solar = cost_lcoe[~mask]

### **Visualize Results**

In [ ]:
datasets = [
    {
    'df': cost_lcoe,
    'value_column': 'value_converted',
    'title': 'c) PV',
    'plot_mode': 'choropleth',
    'min_datapoints': 3,
    },
    {
    'df': cost_lcoe,
    'value_column': 'value_converted',
    'title': 'd) PV',
    'plot_mode': 'points',
    'spatial_extent': [-20, 60, 5, 40],
    },
    {
    'df': cost_lcoe,
    'value_column': 'value_converted',
    'title': 'c) PV',
    'plot_mode': 'size',
    'ghi_data_path': "../data/visualisation_files/ghi/country_GHI_mean.xlsx",
    'highlight_countries': ['CHN', 'USA', 'SAU', 'MOZ', 'CAN'],
    'x_range': [3,7],
    #'min_datapoints': 5,
    #'show_regression': True,
    },
]

fig, data_tables = visualisation.plot_multiple_datasets_on_world(
    datasets,
    cmap="viridis_r",
    colorbar_title="LCOE [$/MWh]",
    figsize=(26, 12),
    jitter_strength=0,
    default_point_size=150,
    font_size=30,
    #color_scale_range=(0, 520),
    #colorbar_ticks=[0, 100, 200, 300, 400, 500],
    iqr_lower_quantile=0.25,
    iqr_upper_quantile=0.75,
    iqr_multiplier=3,
)

#fig.savefig("../data/figures/results/spatial/lcoe_pv_new2.pdf", bbox_inches='tight')

In [ ]:
data_tables['c) PV']

### **Number of Scenarios**

In [ ]:
df = pd.read_parquet("../data/visualisation_files/df_technologies_nan.parquet")
number_scenarios = df[df["property_mapping"] == "Number of Scenarios"]
number_scenarios = number_scenarios[number_scenarios['unit_normalized'].apply(
    lambda lst: len(lst) == 1 and lst[0].get('unit_uri') in ['http://qudt.org/vocab/unit/UNITLESS']
)]
number_scenarios["unit_normalized"].value_counts()

In [ ]:
number_scenarios = visualisation.convert_spatial(number_scenarios, spatial_col="spatial_scope")
print(number_scenarios["property_base_mapping"].value_counts())

In [ ]:
datasets = [
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'b',
    'plot_mode': 'choropleth',
    'min_datapoints': 3,
    },
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'c',
    'plot_mode': 'points',
    'spatial_extent': [-125, -92, 18, 43],
    },
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'd',
    'plot_mode': 'points',
    'spatial_extent': [118.0, 123.5, 20.0, 41.0],
    },
    # {
    # 'df': number_scenarios,
    # 'value_column': 'value_normalized',
    # 'title': 'e',
    # 'plot_mode': 'points',
    # 'spatial_extent': [0, 15, 41.0, 51],
    # },
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'e',
    'plot_mode': 'points',
    'spatial_extent': [38, 61, 20, 38],
    },
]

fig, data_tables = visualisation.plot_multiple_datasets_on_world(
    datasets,
    colorbar_title="Number of Scenarios",
    figsize=(26, 12),
    jitter_strength=0,
    default_point_size=300,
    font_size=30,
    color_scale_range=(2, 8),
    colorbar_ticks=[3, 5, 7],
    iqr_lower_quantile=0.25,
    iqr_upper_quantile=0.75,
    iqr_multiplier=3,
)

fig.savefig("../data/figures/results/spatial/number_scenarios.pdf", bbox_inches='tight')